# Example: Beam focusing into a slab of crystal

Compute the waist diameter inside the crystal and the confocal parameter (collimated range).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from gbeampro import (
    GaussBeam, OpticalSystem,
    Propagation, ThinLens, Interface,
)
import gbeampro.plot as gplot
import gbeampro.analysis as ga
import gbeampro
print('gbeampro version:', gbeampro.__version__)

## Setup

In [ ]:
# LBO crystal at 1064 nm (ordinary, ~149 degC)
n_LBO = 1.604333
L_crystal = 20.0  # mm

beam = GaussBeam.from_waist(wl_um=1.064, w0_mm=1.5)

sys = (OpticalSystem()
       .add(ThinLens(f_mm=150))
       .add(Propagation(145))
       .add(Interface(n1=1.0, n2=n_LBO))
       .add(Propagation(L_crystal))
       .add(Interface(n1=n_LBO, n2=1.0))
       .add(Propagation(195))
       .add(ThinLens(f_mm=200))
       .add(Propagation(100)))

print(sys)

## Trace & plot

In [ ]:
traj = sys.trace(beam, dz=0.5)

fig, ax = plt.subplots(figsize=(12, 4))
gplot.plot_system(sys, traj, ax, label='1.064 µm')
ax.set_title('Beam focusing into LBO crystal')
plt.tight_layout()

## Beam state at each element

In [ ]:
print(sys.summary(beam))

## Beam waists

In [ ]:
waists = ga.find_waists(traj)
for i, w in enumerate(waists):
    zR = ga.rayleigh_range(w)
    print(f'No.{i}:')
    print(f'  Waist location,  z       : {w.z_mm:.3f} mm')
    print(f'  Refractive index, n      : {w.n:.4f}')
    print(f'  Waist diameter,  2*w0    : {w.w_mm * 2e3:.1f} µm')
    print(f'  Confocal parameter, 2*zR : {ga.confocal_parameter(w):.2f} mm')

## w, R, n, θ plots

In [ ]:
zs = np.array([b.z_mm for b in traj])
ws = np.array([b.w_mm * 1e3 for b in traj])
Rs = np.array([b.R_mm for b in traj])
ns = np.array([b.n for b in traj])
ths = np.array([b.theta * 1e3 for b in traj])  # mrad

fig, axes = plt.subplots(4, 1, sharex=True, figsize=(10, 10))

axes[0].plot(zs, ns);               axes[0].set_ylabel('$n$')
axes[1].plot(zs, ws);               axes[1].set_ylabel('$w$ (µm)')
axes[2].plot(zs, Rs);               axes[2].set_ylabel('$R$ (mm)'); axes[2].set_yscale('symlog')
axes[3].semilogy(zs, ths * 1e3);    axes[3].set_ylabel(r'$\theta$ (µrad)')

for ax in axes:
    ax.set_xlabel('$z$ (mm)')

plt.tight_layout()